# Q2 — dbutils.widgets: Parametrização de Notebooks

**Semana 2 | Spark no Databricks**

Em produção, notebooks raramente rodam com parâmetros hardcoded. `dbutils.widgets` é a forma nativa do Databricks de parametrizar notebooks — funciona tanto interativamente (dropdown/input na UI) quanto programaticamente (via Jobs ou `%run` passando parâmetros).

**Tipos de widget disponíveis:**

| Tipo | Comando | Uso |
|------|---------|-----|
| Texto livre | `dbutils.widgets.text()` | Datas, nomes, paths |
| Dropdown fixo | `dbutils.widgets.dropdown()` | Valores pré-definidos |
| Combobox | `dbutils.widgets.combobox()` | Pré-definidos + texto livre |
| Multiselect | `dbutils.widgets.multiselect()` | Múltipla seleção |

> ⚙️ **Execute no Databricks com Serverless.** Os widgets aparecem como controles visuais no topo do notebook.

---
## Parte 1 — Criar os Widgets

Execute a célula abaixo **uma vez** — os controles aparecem no topo do notebook. Depois altere os valores nos controles antes de continuar.

In [ ]:
# Limpar widgets anteriores (útil ao re-executar o notebook)
dbutils.widgets.removeAll()

# Widget de texto — aceita qualquer string
dbutils.widgets.text("data_inicio", "2026-01-01", "Data de início (YYYY-MM-DD)")
dbutils.widgets.text("data_fim",    "2026-12-31", "Data de fim (YYYY-MM-DD)")

# Dropdown — apenas valores da lista são aceitos
dbutils.widgets.dropdown(
    "categoria", "eletronicos",
    ["eletronicos", "roupas", "alimentos"],
    "Categoria de produto"
)

# Combobox — valores sugeridos + texto livre
dbutils.widgets.combobox(
    "formato_saida", "delta",
    ["delta", "parquet", "csv"],
    "Formato de saída"
)

print("Widgets criados! Ajuste os valores no topo do notebook antes de continuar.")

In [ ]:
# Ler os valores dos widgets — sempre strings, mesmo que pareça número/data
data_inicio  = dbutils.widgets.get("data_inicio")
data_fim     = dbutils.widgets.get("data_fim")
categoria    = dbutils.widgets.get("categoria")
formato      = dbutils.widgets.get("formato_saida")

print(f"Parâmetros recebidos:")
print(f"  data_inicio  : {data_inicio}  (tipo: {type(data_inicio).__name__})")
print(f"  data_fim     : {data_fim}")
print(f"  categoria    : {categoria}")
print(f"  formato      : {formato}")
print(f"\nProcessando: {categoria} | {data_inicio} → {data_fim} | formato: {formato}")

---
## Parte 2 — Gerar e filtrar dados com os parâmetros

In [ ]:
import random
from datetime import date, timedelta
from pyspark.sql import functions as F

# Gerar 500 pedidos aleatórios com seed fixo para reprodutibilidade
random.seed(42)

pedidos = [
    (
        i,
        random.choice(["eletronicos", "roupas", "alimentos"]),
        round(random.uniform(10.0, 5000.0), 2),
        (date(2026, 1, 1) + timedelta(days=random.randint(0, 364))).isoformat()
    )
    for i in range(1, 501)
]

df_pedidos = spark.createDataFrame(pedidos, ["id", "categoria", "valor", "data"])

print(f"Total gerado: {df_pedidos.count()} pedidos")
print(f"Distribuição por categoria:")
df_pedidos.groupBy("categoria").count().orderBy("categoria").show()

In [ ]:
# Filtrar usando os valores dos widgets
# .between() no Spark funciona com strings de data no formato ISO (YYYY-MM-DD)
df_filtrado = (
    df_pedidos
    .filter(F.col("categoria") == categoria)
    .filter(F.col("data").between(data_inicio, data_fim))
    .orderBy("data")
)

total_filtrado = df_filtrado.count()
print(f"Registros após filtro ({categoria}, {data_inicio} → {data_fim}): {total_filtrado}")

display(df_filtrado)

---
## Parte 3 — Salvar no formato escolhido pelo widget

In [ ]:
# Setup: criar o volume Unity Catalog para escrita
# DBFS root (dbfs:/tmp/) está desabilitado neste workspace — usar /Volumes/ para escrita
# Catálogo disponível neste trial: 'workspace' (não 'main')
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.estudos")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.estudos.semana02_tmp")
print("Volume pronto: /Volumes/workspace/estudos/semana02_tmp/")

In [ ]:
# Salvar no formato escolhido pelo widget
# Unity Catalog Volume — catálogo 'workspace', schema 'estudos'
output_path = f"/Volumes/workspace/estudos/semana02_tmp/{categoria}_{formato}"

if formato == "delta":
    df_filtrado.write.format("delta").mode("overwrite").save(output_path)
elif formato == "parquet":
    df_filtrado.write.parquet(output_path, mode="overwrite")
elif formato == "csv":
    df_filtrado.write.csv(output_path, mode="overwrite", header=True)
else:
    raise ValueError(f"Formato não suportado: {formato}")

print(f"Dados salvos em: {output_path}")
print(f"Arquivos gerados:")
display(dbutils.fs.ls(output_path))

In [ ]:
# Limpeza — remover os dados temporários após verificação
dbutils.fs.rm(f"/Volumes/workspace/estudos/semana02_tmp/{categoria}_{formato}", recurse=True)
print("Dados temporários removidos.")